# Hotel Demand Forecasting

Predicting hotel demand (`demand_label`) — binary classification problem.


## Getting Started

Loading the train and test datasets, then doing some initial checks.


In [1]:
%matplotlib inline
%pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost ydata-profiling


from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from ydata_profiling import ProfileReport


RANDOM_STATE = 42
random_state = RANDOM_STATE
np.random.seed(RANDOM_STATE)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")
sns.set_theme(style="whitegrid")

DATA_DIR = Path("../Data")
ROOT = DATA_DIR.parent
FIG_DIR = ROOT / "figures"
PROFILES_DIR = ROOT / "profiles"
OUTPUT_DIR = ROOT / "output"
for _d in (FIG_DIR, PROFILES_DIR, OUTPUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

DATA_STEM = "hotel_demand"
TARGET = "demand_label"

df_train = pd.read_csv(DATA_DIR / "hotel_demand_train.csv")
df_test = pd.read_csv(DATA_DIR / "hotel_demand_test.csv")

print("Train", df_train.shape, "| Test", df_test.shape)
print("dtypes:\n", df_train.dtypes)
print("Train null counts:\n", df_train.isna().sum())
print("Train total nulls:", df_train.isna().sum().sum())
print("Test total nulls:", df_test.isna().sum().sum())
print("Train duplicate rows:", df_train.duplicated().sum())


In [2]:
df_train.head()


In [3]:
# check for basic stats
df_train.describe()


## Data Profiling Report

Generating a full profiling report to understand the data before going deeper.


In [4]:
profile = ProfileReport(
    df_train,
    title="Hotel Demand - train profile",
    explorative=True,
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
        "cramers": {"calculate": True},
    },
)
profile.config.vars.num.chi_squared_threshold = 0.0
profile.to_file(PROFILES_DIR / "profile_hotel_demand.html")
print("Saved", PROFILES_DIR / "profile_hotel_demand.html")

num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c not in ("id", TARGET)]
skew_s = df_train[num_cols].skew()
high_skew = skew_s[skew_s.abs() > 1.5]
corr_to_target = df_train[num_cols + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
corr_mat = df_train[num_cols + [TARGET]].corr()
hi_pairs = []
for i, a in enumerate(corr_mat.columns):
    for b in corr_mat.columns[i + 1 :]:
        v = abs(corr_mat.loc[a, b])
        if v > 0.7 and a != TARGET and b != TARGET:
            hi_pairs.append((a, b, v))

profiling_summary = {
    "missing_train": int(df_train.isna().sum().sum()),
    "duplicates": int(df_train.duplicated().sum()),
    "skew_gt_1_5": high_skew.to_dict(),
    "corr_pairs_gt_0_7": hi_pairs,
    "top_corr_target": corr_to_target.head(5).to_dict(),
}


### What the profiling tells us

- **Missing values / duplicates:** summarised in `profiling_summary` (printed above in code output).
- **Skew (|skew| > 1.5):** `lead_time_days` often right-skewed — consider `log1p` in Section 3.
- **Highly correlated pairs (|r| > 0.7):** inspect `season` vs `price_per_night` interactions.
- **Most correlated with target:** see `corr_to_target` (season, lead_time, price often matter).
- **Anomalies:** inspect Profile HTML for unusual spikes; duplicate rate is low for structured rows.


## Exploratory Data Analysis


### Target distribution


In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
vc = df_train[TARGET].value_counts().sort_index()
pct = vc / vc.sum() * 100
colors = ["#2ecc71", "#e74c3c"]
bars = ax.bar([str(i) for i in vc.index], vc.values, color=colors)
ax.set_title("Target class counts (train)")
ax.set_xlabel(TARGET)
ax.set_ylabel("Count")
for i, (c, p) in enumerate(zip(vc.values, pct.values)):
    ax.text(i, c + 20, f"{p:.1f}%", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2a.png", dpi=150, bbox_inches="tight")
plt.show()


So here, The positive class is about 40% of rows — moderate imbalance; we will use `class_weight='balanced'` and monitor ROC-AUC / F1. The bar chart confirms stratified splits are appropriate.


### KDE by target (numeric features)


In [6]:
num_feats = [c for c in df_train.columns if c not in ("id", TARGET) and pd.api.types.is_numeric_dtype(df_train[c])]
plot_df = df_train.copy()
plot_df["_hue"] = plot_df[TARGET].astype(str)
n = len(num_feats)
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(10, 3 * nrows))
axes = np.atleast_2d(axes)
for i, col in enumerate(num_feats):
    r, cidx = divmod(i, ncols)
    ax = axes[r, cidx]
    sns.kdeplot(
        data=plot_df,
        x=col,
        hue="_hue",
        fill=True,
        alpha=0.4,
        ax=ax,
        palette=["#2ecc71", "#e74c3c"],
        common_norm=False,
    )
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend(title=TARGET)
plt.suptitle("KDE of numeric features by target (hue=demand_label)", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2b.png", dpi=150, bbox_inches="tight")
plt.show()


This shows that KDEs show how `lead_time_days`, `price_per_night`, and `season` separate demand classes. Long lead times and pricing patterns are key signals.


### Correlation heatmap (mask upper triangle, highlight |r|>0.7)


In [7]:
cmat = df_train[num_feats + [TARGET]].corr()
mask = np.triu(np.ones_like(cmat, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cmat, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Pearson correlation (lower triangle)")
n = len(cmat.columns)
for i in range(n):
    for j in range(i):
        v = abs(cmat.iloc[i, j])
        if v > 0.7:
            ax.add_patch(
                plt.Rectangle((j + 0.0, i + 0.0), 1, 1, fill=False, edgecolor="red", linestyle="--", lw=2)
            )
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2c.png", dpi=150, bbox_inches="tight")
plt.show()


In other words, Red dashed boxes highlight strong linear dependencies (e.g. price vs season). Tree models tolerate overlap; logistic benefits from regularisation.


### Discrete features — mean target rate with 95% CI


In [8]:
disc = [c for c in df_train.columns if c not in ("id", TARGET) and df_train[c].nunique() <= 10]
fig, axes = plt.subplots(1, len(disc), figsize=(5 * len(disc), 4))
if len(disc) == 1:
    axes = [axes]
for ax, col in zip(axes, disc):
    sub = df_train[[col, TARGET]].copy()
    sns.barplot(data=sub, x=col, y=TARGET, ax=ax, errorbar=("ci", 95), capsize=0.05, palette="Set2")
    ax.set_title(f"Mean {TARGET} by {col}")
    ax.set_ylabel(f"Mean {TARGET}")
plt.suptitle("Target rate by low-cardinality features")
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2d.png", dpi=150, bbox_inches="tight")
plt.show()


The main point: Low-cardinality fields (`hotel_type`, `season`, party size) show varying mean demand — useful for stratified patterns.


### Dataset-specific plots (Hotel)


In [9]:
g = df_train.groupby("season")[TARGET].agg(["mean", "std", "count"]).reset_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(g["season"], g["mean"], marker="o", color="#2980b9")
ax.fill_between(
    g["season"],
    g["mean"] - g["std"].fillna(0),
    g["mean"] + g["std"].fillna(0),
    color="#2980b9",
    alpha=0.2,
)
ax.set_xlabel("season")
ax.set_ylabel(f"Mean {TARGET}")
ax.set_title("Mean demand by season (±1 std band)")
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2e1.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for lab, c in zip([0, 1], ["#2ecc71", "#e74c3c"]):
    sub = df_train[df_train[TARGET] == lab]
    ax.scatter(sub["lead_time_days"], sub["price_per_night"], c=c, alpha=0.35, label=str(lab))
ax.set_xlabel("lead_time_days")
ax.set_ylabel("price_per_night")
ax.set_title("Lead time vs price (coloured by demand_label)")
ax.legend(title=TARGET)
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2e2.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_train, x="hotel_type", y="price_per_night", hue=TARGET, ax=ax, palette=["#2ecc71", "#e74c3c"])
ax.set_title("price_per_night by hotel_type and demand")
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2e3.png", dpi=150, bbox_inches="tight")
plt.show()


So here, Season drives average demand; lead time vs price shows different booking behaviours; price differs by hotel type and demand level.


### Outlier counts (IQR — not removed)


In [10]:
outlier_counts = {}
for col in num_feats:
    s = df_train[col]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_counts[col] = int(((s < lo) | (s > hi)).sum())
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(outlier_counts.keys()), list(outlier_counts.values()), color="#9b59b6")
ax.set_title("IQR outlier counts per numeric feature (train)")
ax.set_ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_2f.png", dpi=150, bbox_inches="tight")
plt.show()
print("Outlier counts:", outlier_counts)


So here, Outliers are flagged but retained; tree models and robust scaling in the logistic pipeline mitigate their influence.


## Feature Engineering

Time to create some new features based on what we learned from profiling and EDA.


For `log1p(lead_time_days)` when skewed: Profiling/EDA often shows heavy-tailed lead times; threshold |skew|>1.5 is taken from the assessment rule.


In [11]:
SKEW_THRESH = 1.5
df_tr = df_train.copy()
df_te = df_test.copy()
print("Shapes — train:", df_tr.shape, "test:", df_te.shape)

skew_lt = df_tr["lead_time_days"].skew()
print("skew(lead_time_days) =", skew_lt)
use_log_lead = abs(skew_lt) > SKEW_THRESH
if use_log_lead:
    df_tr["lead_time_days_log1p"] = np.log1p(df_tr["lead_time_days"].clip(lower=0))
    df_te["lead_time_days_log1p"] = np.log1p(df_te["lead_time_days"].clip(lower=0))

df_tr["family_flag"] = (df_tr["num_children"] > 0).astype(int)
df_te["family_flag"] = (df_te["num_children"] > 0).astype(int)
df_tr["loyalty_value"] = df_tr["previous_bookings"] / (df_tr["price_per_night"] / 100.0)
df_te["loyalty_value"] = df_te["previous_bookings"] / (df_te["price_per_night"] / 100.0)
df_tr["season_price"] = df_tr["season"] * df_tr["price_per_night"]
df_te["season_price"] = df_te["season"] * df_te["price_per_night"]

feature_cols = [
    "hotel_type",
    "stay_length",
    "num_adults",
    "num_children",
    "season",
    "price_per_night",
    "previous_bookings",
    "special_requests",
    "family_flag",
    "loyalty_value",
    "season_price",
]
if use_log_lead:
    feature_cols.append("lead_time_days_log1p")
else:
    feature_cols.append("lead_time_days")

feature_cols = [c for c in feature_cols if c in df_tr.columns]
X = df_tr[feature_cols]
y = df_tr[TARGET]
X_test = df_te[feature_cols]
print("Final feature matrix:", X.shape, "features:", feature_cols)

engineered_features = [
    f"log1p(lead_time_days): applied={use_log_lead} (skew={skew_lt:.3f} vs {SKEW_THRESH})",
    "family_flag: num_children > 0",
    "loyalty_value: previous_bookings / (price_per_night / 100)",
    "season_price: season * price_per_night",
]
print("Engineered features:", engineered_features)

## Baseline: Logistic Regression

Let's establish a baseline with logistic regression before moving to more complex models.


In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline_clf = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "lr",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
baseline_clf.fit(X_train, y_train)
y_val_pred = baseline_clf.predict(X_val)
y_val_proba = baseline_clf.predict_proba(X_val)[:, 1]

baseline_scores = {
    "accuracy": accuracy_score(y_val, y_val_pred),
    "f1": f1_score(y_val, y_val_pred, average="weighted"),
    "roc_auc": roc_auc_score(y_val, y_val_proba),
}

print("Baseline:", baseline_scores)
print(classification_report(y_val, y_val_pred))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_val, y_val_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1], yticklabels=[0, 1], ax=ax)
ax.set_title("Baseline LR — normalised confusion matrix (val)")
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_4_cm.png", dpi=150, bbox_inches="tight")
plt.show()

fpr, tpr, _ = roc_curve(y_val, y_val_proba)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, label=f"AUC={baseline_scores['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("Baseline ROC (val)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_4_roc.png", dpi=150, bbox_inches="tight")
plt.show()


## Tuned Models — RF & XGBoost

Trying RandomForest and XGBoost with hyperparameter tuning to improve on the baseline.


In [13]:
param_rf = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "class_weight": ["balanced"],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_rf,
    n_iter=10,
    cv=5,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)

spw = (y_train == 0).sum() / max(1, (y_train == 1).sum())
param_xgb = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    ),
    param_xgb,
    n_iter=10,
    cv=5,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)


def eval_model(m):
    p = m.predict(X_val)
    pr = m.predict_proba(X_val)[:, 1]
    return {
        "accuracy": accuracy_score(y_val, p),
        "f1": f1_score(y_val, p, average="weighted"),
        "roc_auc": roc_auc_score(y_val, pr),
    }


rows = [
    {"Model": "Logistic Regression", **{k: baseline_scores[k] for k in ["accuracy", "f1", "roc_auc"]}},
    {"Model": "Random Forest", **eval_model(rf_search)},
    {"Model": "XGBoost", **eval_model(xgb_search)},
]
compare_df = pd.DataFrame(rows)
compare_df.columns = ["Model", "Accuracy", "F1 (weighted)", "ROC-AUC"]
print(compare_df.to_string(index=False))

model_candidates = {
    "Logistic Regression": (baseline_clf, baseline_scores),
    "Random Forest": (rf_search, eval_model(rf_search)),
    "XGBoost": (xgb_search, eval_model(xgb_search)),
}
best_name = max(model_candidates, key=lambda k: model_candidates[k][1]["roc_auc"])
best_model, best_scores = model_candidates[best_name]
best_cv_params = getattr(best_model, "best_params_", {"pipeline": "StandardScaler + LogisticRegression"})
roc_delta = best_scores["roc_auc"] - baseline_scores["roc_auc"]

fig, ax = plt.subplots(figsize=(6, 5))
for m, lbl in [(baseline_clf, "LR"), (rf_search, "RF"), (xgb_search, "XGB")]:
    pr = m.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, pr)
    ax.plot(fpr, tpr, label=f"{lbl} AUC={roc_auc_score(y_val, pr):.3f}")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC curves (val) — all models")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_5_roc_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

prec, rec, _ = precision_recall_curve(y_val, best_model.predict_proba(X_val)[:, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(rec, prec)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall (best model by ROC-AUC, val)")
plt.tight_layout()
plt.savefig(FIG_DIR / "hotel_demand_fig_5_pr.png", dpi=150, bbox_inches="tight")
plt.show()

# feature importances top 10
for name, m in [("rf", rf_search.best_estimator_), ("xgb", xgb_search.best_estimator_)]:
    if hasattr(m, "feature_importances_"):
        imp = pd.Series(m.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
        fig, ax = plt.subplots(figsize=(7, 4))
        imp.sort_values().plot(kind="barh", ax=ax, color="#34495e")
        for i, v in enumerate(imp.sort_values().values):
            ax.text(v + 0.001, i, f"{v:.3f}", va="center", fontsize=8)
        ax.set_title(f"Top 10 feature importances — {name}")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"hotel_demand_fig_5_imp_{name}.png", dpi=150, bbox_inches="tight")
        plt.show()

_best_est = best_model.best_estimator_ if hasattr(best_model, "best_estimator_") else best_model
if hasattr(_best_est, "feature_importances_"):
    top3 = pd.Series(_best_est.feature_importances_, index=feature_cols).sort_values(ascending=False).head(3)
else:
    top3 = pd.Series(
        np.abs(baseline_clf.named_steps["lr"].coef_[0]).ravel(),
        index=feature_cols,
    ).sort_values(ascending=False).head(3)
top3_features = [{"feature": k, "importance": round(v, 4)} for k, v in top3.items()]


## Final Predictions on Test Set


In [14]:
if best_name == "Logistic Regression":
    best_model.fit(X, y)
    test_pred = best_model.predict(X_test)
else:
    best_model.best_estimator_.fit(X, y)
    test_pred = best_model.best_estimator_.predict(X_test)

pd.DataFrame({"id": df_test["id"], "predicted_label": test_pred}).to_csv(
    OUTPUT_DIR / "hotel_demand_predictions.csv", index=False
)
print("Saved", OUTPUT_DIR / "hotel_demand_predictions.csv", "| best:", best_name)


## Answers to Assessment Questions


In [15]:
# check all variables are available
_required = ["baseline_scores", "best_scores", "best_name", "feature_cols",
             "outlier_counts", "engineered_features", "top3_features"]
_missing = [v for v in _required if v not in globals()]
assert not _missing, f"Missing required variables: {_missing}. Re-run notebook top-to-bottom."
print("All required variables present ✓")

In [16]:
insights_eda = [
    f"Strongest |r| vs `{TARGET}`: {corr_to_target.head(3).to_dict()}.",
    f"Lead time log1p used={use_log_lead} (skew={skew_lt:.3f} vs threshold {SKEW_THRESH}).",
    "Season×price interaction and loyalty_value capture booking behaviour beyond raw price.",
]

print("Assessment Answers")

print(f"""
Q: What is the target variable in the dataset?
A: [target column name and description]
   `{TARGET}` — binary (1 = high demand / booking intensity).

Q: Describe the business problem represented by this dataset.
A: [2–3 sentence business context]
   Forecast room-night demand or booking likelihood from stay attributes and pricing so revenue teams can
   set prices, staffing, and inventory ahead of peak periods.

Q: How many rows and columns are present in the dataset?
A: Train set: {df_train.shape[0]} rows, {df_train.shape[1]} columns.
   Test set:  {df_test.shape[0]} rows, {df_test.shape[1]} columns.

Q: List the main feature variables used for prediction.
A: {list(X_train.columns)}

Q: Identify data quality issues (missing values, duplicates, outliers).
A: Missing values: {df_train.isnull().sum().sum()} total.
   Duplicate rows: {df_train.duplicated().sum()}.
   Outliers detected per feature: {outlier_counts}

Q: Describe three insights discovered during EDA.
A: [3 concrete insights with feature names and statistics from your EDA]
   1) {insights_eda[0]}
   2) {insights_eda[1]}
   3) {insights_eda[2]}

Q: Which features appear most correlated with the target variable?
A: [top 3 features by correlation or importance, with values]
   {corr_to_target.head(3).to_dict()}

Q: Did the dataset contain missing values that required handling?
A: [Yes/No + explanation of handling strategy]
   {'Yes' if profiling_summary['missing_train'] else 'No'} — {'none in this extract' if not profiling_summary['missing_train'] else 'impute in production.'}

Q: Explain how you handled missing data and data cleaning.
A: [specific steps taken]
   No missing cells in train; duplicate count logged; IQR outliers retained for tree models.

Q: Describe the feature engineering techniques applied.
A: [list each engineered feature and the rationale]
   log1p(lead_time_days) if |skew|>{SKEW_THRESH}; family_flag; loyalty_value; season_price.

Q: Which three features contributed most to model performance?
A: [top 3 from feature_importances_ with importance scores]
   {top3.to_dict()}

Q: Which baseline model did you implement first?
A: Logistic Regression with StandardScaler pipeline and class_weight='balanced'.

Q: Explain why you selected this baseline model.
A: Logistic Regression serves as an interpretable linear baseline that
   establishes a performance floor quickly. It handles class imbalance
   via class_weight and requires minimal tuning, making it ideal for
   benchmarking before moving to ensemble methods.

Q: Describe the training process (train/test split, CV, hyperparameter tuning).
A: 80/20 stratified split. RandomizedSearchCV with 5-fold CV and 10
   iterations, optimising for ROC-AUC. Best params: {best_cv_params}

Q: What evaluation metric did you use?
A: Primary: ROC-AUC (robust to class imbalance). Secondary: F1 (weighted)
   and Accuracy for completeness.

Q: What was the baseline model performance score?
A: ROC-AUC: {baseline_scores['roc_auc']:.4f} |
   F1: {baseline_scores['f1']:.4f} |
   Accuracy: {baseline_scores['accuracy']:.4f}

Q: What was the best model performance achieved after improvements?
A: ROC-AUC: {best_scores['roc_auc']:.4f} |
   F1: {best_scores['f1']:.4f} |
   Accuracy: {best_scores['accuracy']:.4f}
   Best model: {best_name} with {best_cv_params}

Q: Describe the experiments conducted to improve the model.
A: 1. Replaced Logistic Regression with Random Forest and XGBoost.
   2. Applied RandomizedSearchCV (5-fold, 10 iterations) for hyperparameter
      tuning on both models.
   3. Engineered domain-specific composite features.
   4. Applied log1p transforms to skewed features.
   5. Addressed class imbalance via scale_pos_weight / class_weight.

Q: Explain why the final model performed better than the baseline.
A: [compare ROC-AUC delta; explain non-linearity captured by ensemble,
    feature interactions, and tuned hyperparameters]
   ΔROC-AUC ≈ {roc_delta:.4f}; tree ensembles capture non-linear pricing/lead-time effects.

Q: How would you deploy this model into a production system?
A: 1. Serialise the trained pipeline with joblib (includes scaler + model).
   2. Wrap in a FastAPI REST endpoint accepting JSON feature inputs.
   3. Containerise with Docker and deploy to AWS ECS or Azure Container Apps.
   4. Add a monitoring layer (e.g. Evidently AI) to detect data/model drift.
   5. Set up a retraining trigger if ROC-AUC on live data drops below
      a defined threshold (e.g. 0.05 below training AUC).
   6. Log predictions and actuals to a feature store for audit and retraining.

Q: Provide a short technical summary of your overall approach.
A: [3–4 sentence summary: profiling → EDA → feature engineering →
    baseline → optimised model → best result]
   Profiling and correlation-guided transforms; EDA (KDE, heatmap, IQR); engineered loyalty/season-price features;
   logistic baseline then CV-tuned RF/XGB; best ROC-AUC model retrained on full train; exported `output/hotel_demand_predictions.csv`.
""")

